In [1]:
!pip install -U ultralytics timm thop pandas matplotlib openpyxl


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 73.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 81.4 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.0
    Uninstalling matplotlib-3.10.0:
      Successfully uninstalled matplotlib-3.10.0
  Attempting uninstall: timm
    Found existing installation: timm 1.0.20
    Uninstalling timm-1.0.20:
      Successfully uninstalled timm-1.0.20
ERROR: pip's dependency resolver d

In [2]:
from ultralytics import YOLO

model = YOLO("yolo11m.pt")
model.info()


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLO11m summary: 231 layers, 20,114,688 parameters, 0 gradients, 68.5 GFLOPs


(231, 20114688, 0, 68.52838399999999)

In [3]:
import yaml

cfg = model.model.yaml

with open("yolov11m_eca.yaml", "w") as f:
    yaml.dump(cfg, f)

print("Extracted YOLOv11m YAML from official checkpoint")


Extracted YOLOv11m YAML from official checkpoint


In [4]:
with open("yolov11m_eca.yaml", "r") as f:
    cfg = yaml.safe_load(f)

replaced = False
for i, layer in enumerate(cfg["backbone"]):
    if layer[2] == "C2PSA":
        cfg["backbone"][i] = [-1, 1, "ECA", [layer[3][0]]]
        replaced = True
        print("Replaced C2PSA → ECA")

assert replaced, "C2PSA not found!"

with open("yolov11m_eca.yaml", "w") as f:
    yaml.dump(cfg, f)


Replaced C2PSA → ECA


In [5]:
import torch
import torch.nn as nn

class ECA(nn.Module):
    def __init__(self, c, k=3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, k, padding=(k - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)
        y = self.conv(y.squeeze(-1).transpose(-1, -2))
        y = self.sigmoid(y).transpose(-1, -2).unsqueeze(-1)
        return x * y


In [6]:
# 1. Define ECA (you already did this in CELL 5)

# 2. Register ECA into Ultralytics module registry
import ultralytics.nn.modules as modules
modules.ECA = ECA

# 3. Register ECA into Ultralytics task parser globals
import ultralytics.nn.tasks as tasks
tasks.ECA = ECA

print("ECA registered in modules AND tasks (this fixes KeyError)")


ECA registered in modules AND tasks (this fixes KeyError)


In [7]:
import ultralytics.nn.modules as m
print("ECA" in dir(m))

True


In [8]:
%%writefile c2a.yaml
train: /kaggle/input/c2a-dataset/C2A_Dataset/new_dataset3/train/images
val: /kaggle/input/c2a-dataset//C2A_Dataset/new_dataset3/val/images

nc: 1
names: ['person']



Writing c2a.yaml


In [9]:
baseline = YOLO("yolo11m.pt")

baseline.train(
    data="c2a.yaml",
    epochs=50,          # max epochs
    imgsz=640,
    batch=16,
    device=0,
    name="yolo11m_baseline",
    patience=5,         # EARLY STOPPING (important)
    save=True,
    save_period=-1
)


Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c2a.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11m_baseline, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=5, perspective=0.0, plots=True,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a2c84bcc4d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [10]:
eca_model = YOLO("yolov11m_eca.yaml")
eca_model.load("yolo11m.pt")

eca_model.train(
    data="c2a.yaml",
    epochs=50,          # max epochs
    imgsz=640,
    batch=16,
    device=0,
    name="yolo11m_eca",
    patience=5,         # EARLY STOPPING
    save=True,
    save_period=-1
)


Transferred 607/608 items from pretrained weights
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c2a.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov11m_eca.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11m_eca, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a2c20987170>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [11]:
base_metrics = baseline.val(data="c2a.yaml")
eca_metrics = eca_model.val(data="c2a.yaml")
import ultralytics.nn.modules as m
print("ECA" in dir(m))
print("ECA" in dir(m))
print("Baseline:", base_metrics.results_dict)
print("ECA:", eca_metrics.results_dict)


Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 957.6±618.6 MB/s, size: 796.1 KB)
val: Scanning /kaggle/input/c2a-dataset/C2A_Dataset/new_dataset3/val/labels... 2043 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2043/2043 672.8it/s 3.0s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/c2a-dataset/C2A_Dataset/new_dataset3/val is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 128/128 2.1it/s 1:020.5sss
                   all       2043      72123      0.881        0.8      0.852      0.613
Speed: 0.9ms preprocess, 23.0ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
YOLOv11m_eca summary (fused

In [12]:
import os

BASE_DIR = "/kaggle/working"
EXCEL_DIR = f"{BASE_DIR}/excel_reports"
PLOT_DIR = f"{BASE_DIR}/plots"
REPORT_DIR = f"{BASE_DIR}/benchmark_reports"

for d in [EXCEL_DIR, PLOT_DIR, REPORT_DIR]:
    os.makedirs(d, exist_ok=True)


In [13]:
import pandas as pd

BASELINE_RUN = "runs/detect/yolo11m_baseline"
ECA_RUN = "runs/detect/yolo11m_eca"

baseline_df = pd.read_csv(f"{BASELINE_RUN}/results.csv")
eca_df = pd.read_csv(f"{ECA_RUN}/results.csv")


In [14]:
baseline_df.to_csv(f"{EXCEL_DIR}/yolo11m_baseline_training.csv", index=False)
eca_df.to_csv(f"{EXCEL_DIR}/yolo11m_eca_training.csv", index=False)

baseline_df.to_excel(f"{EXCEL_DIR}/yolo11m_baseline_training.xlsx", index=False)
eca_df.to_excel(f"{EXCEL_DIR}/yolo11m_eca_training.xlsx", index=False)


In [ ]:
import matplotlib.pyplot as plt

def plot_losses(df, tag):
    plt.figure(figsize=(12,6))
    for k in ["box", "cls", "dfl"]:
        plt.plot(df["epoch"], df[f"train/{k}_loss"], label=f"Train {k}")
        plt.plot(df["epoch"], df[f"val/{k}_loss"], label=f"Val {k}")
    plt.legend(); plt.grid()
    plt.title(f"{tag} Loss Curves")
    plt.savefig(f"{PLOT_DIR}/{tag}_loss.png", dpi=300)
    plt.close()

plot_losses(baseline_df, "yolo11m_baseline")
plot_losses(eca_df, "yolo11m_eca")


In [ ]:
def plot_metrics(df, tag):
    plt.figure(figsize=(12,6))
    for k in ["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]:
        plt.plot(df["epoch"], df[k], label=k)
    plt.legend(); plt.grid()
    plt.title(f"{tag} Metrics")
    plt.savefig(f"{PLOT_DIR}/{tag}_metrics.png", dpi=300)
    plt.close()

plot_metrics(baseline_df, "yolo11m_baseline")
plot_metrics(eca_df, "yolo11m_eca")


In [ ]:
from datetime import datetime

def write_report(tag, df):
    best = df.loc[df["metrics/mAP50(B)"].idxmax()]
    p, r = best["metrics/precision(B)"], best["metrics/recall(B)"]
    f1 = 2*p*r/(p+r+1e-6)

    report = f"""
================================================================================
BENCHMARK REPORT: {tag}
================================================================================

Best Epoch: {int(best['epoch'])}

Final Losses:
Train - Box {best['train/box_loss']:.4f}, Cls {best['train/cls_loss']:.4f}, DFL {best['train/dfl_loss']:.4f}
Val   - Box {best['val/box_loss']:.4f}, Cls {best['val/cls_loss']:.4f}, DFL {best['val/dfl_loss']:.4f}

Metrics:
Precision {p:.4f}
Recall {r:.4f}
F1 {f1:.4f}
mAP50 {best['metrics/mAP50(B)']:.4f}
mAP50-95 {best['metrics/mAP50-95(B)']:.4f}

Timestamp: {datetime.now()}
================================================================================
"""
    path = f"{REPORT_DIR}/{tag}_benchmark.txt"
    open(path,"w").write(report)
    print("Saved:", path)

write_report("yolo11m_baseline", baseline_df)
write_report("yolo11m_eca", eca_df)


Saved: /kaggle/working/benchmark_reports/yolo11m_baseline_benchmark.txt
Saved: /kaggle/working/benchmark_reports/yolo11m_eca_benchmark.txt


In [ ]:
from IPython.display import FileLink

!zip -r working_directory.zip /kaggle/working
FileLink("working_directory.zip")


  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/plots/ (stored 0%)
  adding: kaggle/working/plots/yolo11m_eca_metrics.png (deflated 16%)
  adding: kaggle/working/plots/yolo11m_eca_loss.png (deflated 12%)
  adding: kaggle/working/plots/yolo11m_baseline_loss.png (deflated 12%)
  adding: kaggle/working/plots/yolo11m_baseline_metrics.png (deflated 15%)
  adding: kaggle/working/yolo11m.pt (deflated 8%)
  adding: kaggle/working/benchmark_reports/ (stored 0%)
  adding: kaggle/working/benchmark_reports/yolo11m_eca_benchmark.txt (deflated 58%)
  adding: kaggle/working/benchmark_reports/yolo11m_baseline_benchmark.txt (deflated 58%)
  adding: kaggle/working/c2a.yaml (deflated 41%)
  adding: kaggle/working/yolo26n.pt (deflated 11%)
  adding: kaggle/working/.virtual_documents/ (stored 0%)
  adding: kaggle/working/.virtual_documents/__notebook_source__.ipynb (deflated 64%)
  adding: kaggle/working/runs/ (stored 0%)
  adding: kaggle/working/runs/detect/ (stored 0%)
  adding: kaggle/wor

/kaggle/working/working_directory.zip